# Protein Domain Plotter

A self-contained toolkit for drawing publication-style **protein domain diagrams**.

**What it does**
- Fetch protein length + domain-like features from **UniProt** — single ID or a comma-separated
  list (`Domain`, `Region`, `Motif`, `Repeat`) — or define proteins manually, or import a
  previously-exported CSV.
- Handle **multiple proteins at once**, plotted stacked, one row per protein.
- Three scale modes: `true` (real aa scale), `equal` (same visual width per protein),
  `normalized` (same width, x-axis in % of length). The axis always starts at **0** unless a
  domain you added explicitly has a negative coordinate.
- True circular rounded corners (max rounding = a real half-circle/pill end), fully customizable
  colors/transparency/edges/fonts/sizes.
- A legend **grouped by protein** (one titled mini-legend per protein).
- Exports high-resolution **PNG**, **JPG**, a vector **SVG** (for Illustrator/Inkscape/etc.), and a
  **CSV** of every domain, each with its own
  download button.
- A compact `ipywidgets` panel (no more, no less than fits your browser width) — plus a plain
  Python API for scripting.

**How to use this notebook**
1. Run the cells in order (top to bottom) once.
2. Use the interactive panel to fetch/add/import proteins, style the plot, and generate the
   figure — **or** call the Python functions directly (see the Quick Start examples at the end).


In [1]:
# 1) Install dependencies (safe to re-run)
!pip install -q requests matplotlib ipywidgets pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.7 MB/s eta 0:00:00


In [2]:
# 2) Imports and core data structures

import csv
import io
import math
import requests
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.legend import Legend
from matplotlib.transforms import blended_transform_factory
from matplotlib.backends.backend_agg import RendererAgg
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Every widget uses this so descriptions are never clipped, regardless of label length.
WSTYLE = {"description_width": "initial"}


class Domain:
    # A single annotated region on a protein (a "domain", "region", "motif", "repeat", or custom).
    def __init__(self, name, start, end, dtype="Domain", color=None, shape=None,
                 alpha=None, edgecolor=None, linewidth=None, corner_radius=None):
        self.name = name
        self.start = int(start)
        self.end = int(end)
        self.dtype = dtype
        # Per-domain style overrides (None = fall back to the global plot settings)
        self.color = color
        self.shape = shape
        self.alpha = alpha
        self.edgecolor = edgecolor
        self.linewidth = linewidth
        self.corner_radius = corner_radius

    def __repr__(self):
        return "Domain(%r, %d-%d, type=%s)" % (self.name, self.start, self.end, self.dtype)


class Protein:
    # A protein: a name, a length (aa), and a list of Domain objects.
    def __init__(self, name, length, accession=None, organism=None):
        self.name = name
        self.length = int(length)
        self.accession = accession
        self.organism = organism
        self.domains = []

    def add_domain(self, domain):
        self.domains.append(domain)
        return domain

    def __repr__(self):
        return "Protein(%r, length=%d, domains=%d)" % (self.name, self.length, len(self.domains))


# Global registry: name -> Protein
PROTEINS = {}


In [3]:
# 3) Fetching from UniProt

UNIPROT_FEATURE_TYPES_DEFAULT = ["Domain", "Region", "Motif", "Repeat"]

# Tracks, per UniProt *recommended name*, which accession is currently using which
# display name -- so that fetching two different proteins (e.g. the same protein
# name in two different species) never silently overwrites one with the other.
_BASE_NAME_REGISTRY = {}

def _disambiguated_name(base_name, accession, organism):
    # Returns the display name to use for `accession`/`base_name`, renaming any
    # earlier same-named-but-different-accession entry (in PROTEINS and in
    # _BASE_NAME_REGISTRY) the first time a real collision is detected.
    existing_map = _BASE_NAME_REGISTRY.setdefault(base_name, {})

    if accession in existing_map:
        return existing_map[accession]          # re-fetching the same accession
    if not existing_map:
        existing_map[accession] = base_name      # first time we've seen this name
        return base_name

    # Collision: `base_name` is already used by >=1 *different* accession.
    suffix = (" [%s]" % organism) if organism else (" (%s)" % accession)
    new_name = base_name + suffix
    existing_map[accession] = new_name

    # If exactly one prior entry is still sitting under the bare `base_name`,
    # rename it too, so every colliding entry ends up consistently suffixed.
    bare_holders = [acc for acc, nm in existing_map.items() if nm == base_name and acc != accession]
    for prior_acc in bare_holders:
        prior_protein = PROTEINS.pop(base_name, None)
        if prior_protein is not None:
            prior_suffix = (" [%s]" % prior_protein.organism) if prior_protein.organism else (" (%s)" % prior_acc)
            prior_new_name = base_name + prior_suffix
            prior_protein.name = prior_new_name
            PROTEINS[prior_new_name] = prior_protein
            existing_map[prior_acc] = prior_new_name
    return new_name


def fetch_uniprot(accession, custom_name=None, feature_types=None, add_to_registry=True):
    # Fetch sequence length + selected feature types from the UniProt REST API
    # and register the protein (unless add_to_registry=False). If the same
    # UniProt *name* is fetched for two different accessions (e.g. the same
    # protein in two species), both entries are automatically disambiguated
    # with an "[Organism]" (or accession) suffix instead of one overwriting
    # the other.
    feature_types = feature_types or UNIPROT_FEATURE_TYPES_DEFAULT
    accession = accession.strip()
    url = "https://rest.uniprot.org/uniprotkb/%s.json" % accession
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    length = data.get("sequence", {}).get("length")
    if length is None:
        raise ValueError("Could not determine sequence length for %s" % accession)

    try:
        rec_name = data["proteinDescription"]["recommendedName"]["fullName"]["value"]
    except (KeyError, TypeError):
        rec_name = accession

    try:
        organism = data["organism"]["scientificName"]
    except (KeyError, TypeError):
        organism = None

    display_name = custom_name if custom_name else _disambiguated_name(rec_name, accession, organism)
    protein = Protein(name=display_name, length=length, accession=accession, organism=organism)

    for feat in data.get("features", []):
        ftype = feat.get("type")
        if ftype not in feature_types:
            continue
        try:
            start = feat["location"]["start"]["value"]
            end = feat["location"]["end"]["value"]
        except (KeyError, TypeError):
            continue
        desc = feat.get("description") or ftype
        protein.add_domain(Domain(name=desc, start=start, end=end, dtype=ftype))

    if add_to_registry:
        PROTEINS[display_name] = protein
    return protein


def fetch_uniprot_multi(accessions, custom_name=None, feature_types=None):
    # Fetch several UniProt IDs at once. `accessions` may be a comma-separated
    # string ("P04637, P00533") or a list of IDs. A custom_name is only applied
    # when exactly one accession is given (otherwise each keeps its UniProt name,
    # auto-disambiguated by organism/accession if two IDs share the same name).
    if isinstance(accessions, str):
        ids = [a.strip() for a in accessions.split(",") if a.strip()]
    else:
        ids = [str(a).strip() for a in accessions if str(a).strip()]

    use_custom = custom_name if len(ids) == 1 else None
    fetched, errors = [], []
    for acc in ids:
        try:
            fetched.append(fetch_uniprot(acc, custom_name=use_custom, feature_types=feature_types))
        except Exception as e:
            errors.append((acc, str(e)))
    return fetched, errors


In [4]:
# 4) Manual protein / domain management, registry helpers, and CSV import/export

def add_protein_manual(name, length):
    # Register a protein by name and total length (aa) with no domains yet.
    protein = Protein(name=name, length=length)
    PROTEINS[name] = protein
    return protein

def add_protein_from_sequence(name, sequence):
    # Register a protein using a raw amino-acid sequence (length is inferred).
    seq = "".join(sequence.split())
    protein = Protein(name=name, length=len(seq))
    PROTEINS[name] = protein
    return protein

def add_domain_to_protein(protein_name, domain_name, start, end, dtype="Domain", **style_kwargs):
    # Add a Domain to an already-registered protein. style_kwargs: color, shape,
    # alpha, edgecolor, linewidth, corner_radius.
    if protein_name not in PROTEINS:
        raise KeyError("Protein '%s' not found. Add it first." % protein_name)
    d = Domain(name=domain_name, start=start, end=end, dtype=dtype, **style_kwargs)
    PROTEINS[protein_name].add_domain(d)
    return d

def remove_protein(name):
    PROTEINS.pop(name, None)

def remove_domain(protein_name, index):
    if protein_name in PROTEINS:
        try:
            del PROTEINS[protein_name].domains[index]
        except IndexError:
            pass

def list_proteins():
    if not PROTEINS:
        print("(no proteins registered yet)")
        return
    for p in PROTEINS.values():
        print(p)
        for i, d in enumerate(p.domains):
            print("   [%d]" % i, d)

def clear_registry():
    PROTEINS.clear()
    _BASE_NAME_REGISTRY.clear()


CSV_FIELDS = ["protein", "length", "domain_name", "start", "end", "type", "color"]

def export_domains_csv(path, protein_names=None):
    # Write every protein (or just protein_names) + their domains to a CSV file.
    names = protein_names or list(PROTEINS.keys())
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writeheader()
        for name in names:
            p = PROTEINS.get(name)
            if p is None:
                continue
            if not p.domains:
                writer.writerow({"protein": p.name, "length": p.length, "domain_name": "",
                                  "start": "", "end": "", "type": "", "color": ""})
            for d in p.domains:
                writer.writerow({"protein": p.name, "length": p.length, "domain_name": d.name,
                                  "start": d.start, "end": d.end, "type": d.dtype,
                                  "color": d.color or ""})
    return path

def import_domains_csv(source, is_content=False):
    # Import proteins/domains from a CSV in the same format export_domains_csv writes
    # (columns: protein, length, domain_name, start, end, type, color). `source` is a
    # file path by default; pass is_content=True to hand it raw text/bytes instead
    # (e.g. straight from a widgets.FileUpload value).
    if is_content:
        text = source.decode("utf-8") if isinstance(source, (bytes, bytearray)) else source
        f = io.StringIO(text)
    else:
        f = open(source, "r", newline="")
    try:
        reader = csv.DictReader(f)
        imported = []
        for row in reader:
            pname = (row.get("protein") or "").strip()
            if not pname:
                continue
            if pname not in PROTEINS:
                length_raw = row.get("length") or 0
                length = int(float(length_raw)) if str(length_raw).strip() else 0
                add_protein_manual(pname, length)
            dname = (row.get("domain_name") or "").strip()
            start_raw, end_raw = row.get("start"), row.get("end")
            if dname and start_raw not in (None, "") and end_raw not in (None, ""):
                add_domain_to_protein(pname, dname, int(float(start_raw)), int(float(end_raw)),
                                      dtype=(row.get("type") or "Domain"),
                                      color=(row.get("color") or None) or None)
            imported.append(pname)
        return sorted(set(imported))
    finally:
        f.close()


In [5]:
# 5) Plotting engine

_KAPPA = 0.5522847498  # cubic-bezier constant for approximating a quarter circle

def _color_lookup(proteins, domains_by_protein, palette, color_map):
    # Build a consistent color per unique domain name, in first-seen order,
    # cycling through the chosen matplotlib colormap. `color_map` (dict of
    # name/dtype -> color) overrides individual entries.
    color_map = color_map or {}
    names_in_order = []
    for p in proteins:
        for d in domains_by_protein[p.name]:
            if d.name not in names_in_order:
                names_in_order.append(d.name)

    cmap = plt.get_cmap(palette)
    n = max(len(names_in_order), 1)
    lookup = {}
    for i, name in enumerate(names_in_order):
        lookup[name] = cmap((i % 20) / 20.0) if palette in ("tab20", "tab20b", "tab20c") \
            else cmap(i / n)

    for key, color in color_map.items():
        lookup[key] = color
    return lookup


def _rounded_rect_path(x0, y0, x1, y1, rx, ry):
    # Build a rounded-rectangle Path whose corners are TRUE circular arcs (approximated
    # with cubic beziers), using independent x/y radii (rx, ry) given in DATA units.
    # This is what lets max rounding render as a real half-circle/pill end, regardless
    # of how different the x- and y-axis data scales are.
    rx = max(0.0, min(rx, (x1 - x0) / 2.0))
    ry = max(0.0, min(ry, (y1 - y0) / 2.0))

    if rx <= 1e-12 or ry <= 1e-12:
        verts = [(x0, y0), (x1, y0), (x1, y1), (x0, y1), (x0, y0)]
        codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.LINETO, Path.CLOSEPOLY]
        return Path(verts, codes)

    kx, ky = _KAPPA * rx, _KAPPA * ry
    verts = [
        (x0 + rx, y0),
        (x1 - rx, y0),
        (x1 - rx + kx, y0), (x1, y0 + ry - ky), (x1, y0 + ry),
        (x1, y1 - ry),
        (x1, y1 - ry + ky), (x1 - rx + kx, y1), (x1 - rx, y1),
        (x0 + rx, y1),
        (x0 + rx - kx, y1), (x0, y1 - ry + ky), (x0, y1 - ry),
        (x0, y0 + ry),
        (x0, y0 + ry - ky), (x0 + rx - kx, y0), (x0 + rx, y0),
        (x0 + rx, y0),
    ]
    codes = [
        Path.MOVETO, Path.LINETO,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.LINETO,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.LINETO,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.LINETO,
        Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.CLOSEPOLY,
    ]
    return Path(verts, codes)


def _draw_box(ax, x0, x1, y0, y1, shape, color, alpha, edgecolor, linewidth, corner_radius, px_per_unit):
    # px_per_unit: (pixels-per-x-data-unit, pixels-per-y-data-unit) of `ax`, used to
    # convert a y-based corner_radius fraction into a matching, visually-circular x radius.
    height = y1 - y0
    if shape == "rect" or corner_radius <= 0:
        patch = Rectangle((x0, y0), x1 - x0, height,
                           facecolor=color, alpha=alpha,
                           edgecolor=edgecolor, linewidth=linewidth)
    else:
        px_x, px_y = px_per_unit
        ry = max(0.0, min(corner_radius, 0.5)) * height
        rx = ry * (px_y / px_x) if px_x > 0 else ry
        path = _rounded_rect_path(x0, y0, x1, y1, rx, ry)
        patch = PathPatch(path, facecolor=color, alpha=alpha,
                           edgecolor=edgecolor, linewidth=linewidth)
    ax.add_patch(patch)


def _draw_grouped_legend(fig, ax, proteins, domains_by_protein, color_lookup, alpha, edgecolor,
                          linewidth, fontsize, area_left, area_right, area_top_y, row_gap_in):
    # One titled mini-legend per protein (grouping domains by the protein they belong
    # to), flowed left-to-right and wrapped to new rows as needed. Uses matplotlib's
    # own legend layout engine (via a throwaway Agg renderer) so wrapping is based on
    # real measured text width, never guessed. Returns (legends, row_heights_in) --
    # row_heights_in lets the caller measure exactly how tall the legend block is,
    # in real inches, independent of whatever figure it was drawn on.
    fig_w_in, fig_h_in = fig.get_size_inches()
    renderer = RendererAgg(int(fig_w_in * fig.dpi) + 2, int(fig_h_in * fig.dpi) + 2, fig.dpi)

    cur_x, cur_y, row_h_in = area_left, area_top_y, 0.0
    gap_x_in = 0.30
    created = []
    row_heights_in = []

    def _place(handles, labels, title, x, y):
        leg = Legend(ax, handles, labels, title=title, loc="upper left",
                     bbox_to_anchor=(x, y), bbox_transform=fig.transFigure,
                     frameon=False, fontsize=fontsize, title_fontsize=fontsize + 1,
                     handlelength=1.3, handleheight=1.0, borderaxespad=0.0,
                     labelspacing=0.35, columnspacing=1.0, handletextpad=0.5)
        leg.get_title().set_fontweight("bold")
        leg.get_title().set_ha("left")
        leg.set_in_layout(True)
        ax.add_artist(leg)
        return leg

    for protein in proteins:
        seen = []
        for d in domains_by_protein[protein.name]:
            if d.name not in seen:
                seen.append(d.name)
        if not seen:
            continue
        handles = [Rectangle((0, 0), 1, 1, facecolor=color_lookup.get(n, "#4C72B0"),
                              alpha=alpha, edgecolor=edgecolor, linewidth=linewidth)
                   for n in seen]

        leg = _place(handles, seen, protein.name, cur_x, cur_y)
        bbox = leg.get_window_extent(renderer).transformed(fig.transFigure.inverted())
        w_in, h_in = bbox.width * fig_w_in, bbox.height * fig_h_in

        if (cur_x + w_in / fig_w_in) > area_right and cur_x > area_left:
            leg.remove()
            row_heights_in.append(row_h_in)
            cur_y -= (row_h_in + row_gap_in) / fig_h_in
            cur_x = area_left
            row_h_in = 0.0
            leg = _place(handles, seen, protein.name, cur_x, cur_y)
            bbox = leg.get_window_extent(renderer).transformed(fig.transFigure.inverted())
            w_in, h_in = bbox.width * fig_w_in, bbox.height * fig_h_in

        row_h_in = max(row_h_in, h_in)
        cur_x += w_in / fig_w_in + gap_x_in / fig_w_in
        created.append(leg)

    if row_h_in > 0 or not row_heights_in:
        row_heights_in.append(row_h_in)
    return created, row_heights_in


def _measure_legend_block_height(fig_width_in, proteins, domains_by_protein, color_lookup,
                                  alpha, edgecolor, linewidth, fontsize, area_left, area_right,
                                  row_gap_in):
    # Dry-runs _draw_grouped_legend on a throwaway figure (tall enough that nothing
    # needs to clip) purely to measure how many rows the legend will need and how
    # tall the whole block is, in real inches -- independent of the final figure's
    # height. This is what lets the final figure be sized tall enough on the first
    # try, instead of guessing and clipping.
    dummy_fig = plt.figure(figsize=(fig_width_in, 12.0))
    dummy_ax = dummy_fig.add_axes([0, 0, 1, 1])
    try:
        _, row_heights_in = _draw_grouped_legend(
            dummy_fig, dummy_ax, proteins, domains_by_protein, color_lookup, alpha,
            edgecolor, linewidth, fontsize, area_left=area_left, area_right=area_right,
            area_top_y=0.95, row_gap_in=row_gap_in)
    finally:
        plt.close(dummy_fig)
    n = len(row_heights_in)
    return sum(row_heights_in) + row_gap_in * max(0, n - 1)


def plot_proteins(protein_names=None,
                   scale_mode="true",             # "true" | "equal" | "normalized"
                   domain_shape="rounded",        # "rounded" | "rect"
                   corner_radius=0.5,             # 0 (square) - 0.5 (full half-circle/pill ends)
                   domain_height=0.6,             # domain box height, as a fraction of row spacing
                   palette="tab20",
                   color_map=None,                # dict: domain name/type -> color, overrides auto colors
                   alpha=0.9,
                   edgecolor="black",
                   linewidth=1.0,
                   backbone_color="#d9d9d9",
                   backbone_edgecolor="#888888",
                   backbone_height_frac=0.35,     # backbone thickness relative to domain box height
                   row_height=1.0,
                   fig_width=14,
                   show_domain_labels=True,
                   show_position_labels=False,    # annotate true aa start/end at every domain edge
                   visible_types=None,             # e.g. ["Domain", "Repeat"] -- None shows everything
                   label_fontsize=8,
                   label_rotation=0,
                   protein_label_fontsize=11,
                   show_length_annotation=True,
                   show_legend=True,
                   title=None,
                   dpi=300,
                   transparent_png=True,
                   save_svg=True,                 # also export a .svg (vector, dpi-independent)
                   save_prefix=None,
                   auto_download=True,
                   show=True):
    # Draw a stacked domain diagram (one row per protein) and, if save_prefix is
    # given, export high-resolution .png and .jpg files (auto-downloaded in Colab
    # when auto_download=True).
    names = protein_names or list(PROTEINS.keys())
    proteins = [PROTEINS[n] for n in names if n in PROTEINS]
    if not proteins:
        raise ValueError("No proteins to plot. Add/fetch some first.")

    # Feature-type filter: build a per-protein list of the domains that should
    # actually be drawn (visible_types=None means "show everything").
    domains_by_protein = {}
    for p in proteins:
        if visible_types is None:
            domains_by_protein[p.name] = p.domains
        else:
            domains_by_protein[p.name] = [d for d in p.domains if d.dtype in visible_types]

    color_lookup = _color_lookup(proteins, domains_by_protein, palette, color_map)
    n_rows = len(proteins)
    box_h = max(0.05, min(domain_height, 0.98))
    y_positions = list(range(n_rows, 0, -1))
    fixed_width = 100.0  # visual width used for "equal" / "normalized" modes
    xlabel_present = scale_mode in ("true", "normalized")
    has_legend_content = show_legend and any(domains_by_protein[p.name] for p in proteins)

    # --- Pre-pass: compute each row's scale/length + the true data extents ------
    # so the axis can start at 0 (never negative, unless a domain explicitly goes
    # negative) and the pixel-ratio used for rounding is correct BEFORE anything
    # is drawn.
    row_data = []
    min_x, max_x = 0.0, 0.0
    for row_y, protein in zip(y_positions, proteins):
        if scale_mode == "true":
            scale, row_len = 1.0, protein.length
        else:
            scale, row_len = fixed_width / protein.length, fixed_width
        row_data.append((row_y, protein, scale, row_len))
        max_x = max(max_x, row_len)
        for d in domains_by_protein[protein.name]:
            min_x = min(min_x, d.start * scale, d.end * scale)
            max_x = max(max_x, d.start * scale, d.end * scale)

    # --- Margins, budgeted in ABSOLUTE INCHES (so short/tall figures both get
    # enough room for text), computed BEFORE the figure is created so the figure
    # can be made exactly tall enough -- including a REAL measurement of the
    # grouped legend's height, not a guess. This is what guarantees the legend
    # (which can need anywhere from 1 to many rows) never gets cramped.
    longest_label_chars = max(
        (len(p.name) + (len("  (%d aa)" % p.length) if show_length_annotation else 0)
         for p in proteins), default=10)
    left_in = 0.15 + longest_label_chars * (protein_label_fontsize * 0.60 / 72.0)
    right_in = 0.25
    top_in = 0.55 if title else 0.15
    xlabel_in = 0.42 if xlabel_present else 0.0
    position_in = 0.28 if show_position_labels else 0.0

    left = min(0.42, left_in / fig_width)
    right = 1.0 - min(0.20, right_in / fig_width)

    legend_gap_in = 0.16
    legend_block_in = 0.0
    if has_legend_content:
        legend_block_in = _measure_legend_block_height(
            fig_width, proteins, domains_by_protein, color_lookup, alpha, edgecolor,
            linewidth, label_fontsize, area_left=left, area_right=right, row_gap_in=0.14)

    bottom_in = 0.15 + xlabel_in + position_in + (legend_gap_in + legend_block_in if has_legend_content else 0.0)
    plot_area_in = max(1.0, row_height * n_rows)
    fig_height = plot_area_in + top_in + bottom_in

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    top = 1.0 - top_in / fig_height
    bottom = bottom_in / fig_height
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom)

    # Axis limits set before drawing so the data<->pixel ratio below is final.
    # Left edge is exactly 0 unless a domain explicitly extends past it -- except
    # we nudge it left by ~1 rendered pixel's worth of data units. Without this,
    # shapes that start exactly at the axis boundary (e.g. every backbone) can
    # have their leftmost antialiased edge clipped by the axes frame, which
    # looks like a flat 1px "cut" instead of a smooth rounded/pill end.
    x_left = min(0.0, min_x)
    x_right = max_x * 1.03
    axes_width_in = (right - left) * fig_width
    one_px_data = (x_right - x_left) / (axes_width_in * dpi) if axes_width_in > 0 and dpi > 0 else 0.0
    ax.set_xlim(x_left - one_px_data, x_right)
    ax.set_ylim(0.3, n_rows + 0.7)

    # Pixels-per-data-unit for x and y, used to make rounded corners truly circular.
    p0 = ax.transData.transform((0, 0))
    p1 = ax.transData.transform((1, 0))
    p2 = ax.transData.transform((0, 1))
    px_per_unit = (abs(p1[0] - p0[0]), abs(p2[1] - p0[1]))

    # Protein-name labels are anchored to the AXES (not the data), just left of the
    # plot area, so they never depend on the data's x-range.
    label_trans = blended_transform_factory(ax.transAxes, ax.transData)

    for row_y, protein, scale, row_len in row_data:
        y0, y1 = row_y - box_h / 2.0, row_y + box_h / 2.0

        bb_y0 = row_y - (box_h * backbone_height_frac) / 2.0
        bb_y1 = row_y + (box_h * backbone_height_frac) / 2.0
        _draw_box(ax, 0, row_len, bb_y0, bb_y1, "rounded", backbone_color, 1.0,
                  backbone_edgecolor, 0.8, 0.5, px_per_unit)

        for d in domains_by_protein[protein.name]:
            x0, x1 = d.start * scale, d.end * scale
            color = d.color or color_lookup.get(d.name, "#4C72B0")
            shape = d.shape or domain_shape
            a = d.alpha if d.alpha is not None else alpha
            ec = d.edgecolor or edgecolor
            lw = d.linewidth if d.linewidth is not None else linewidth
            cr = d.corner_radius if d.corner_radius is not None else corner_radius
            _draw_box(ax, x0, x1, y0, y1, shape, color, a, ec, lw, cr, px_per_unit)

            if show_domain_labels:
                ax.text((x0 + x1) / 2.0, row_y, d.name, ha="center", va="center",
                         fontsize=label_fontsize, rotation=label_rotation, zorder=5)
            if show_position_labels:
                ax.text(x0, y0 - 0.10, str(d.start), ha="center", va="top", fontsize=label_fontsize * 0.75)
                ax.text(x1, y0 - 0.10, str(d.end), ha="center", va="top", fontsize=label_fontsize * 0.75)

        label = protein.name
        if show_length_annotation:
            label += "  (%d aa)" % protein.length
        ax.text(-0.015, row_y, label, transform=label_trans, ha="right", va="center",
                fontsize=protein_label_fontsize, fontweight="bold", clip_on=False)

    if scale_mode == "true":
        ax.set_xlabel("Residue position (aa)", labelpad=10)
    elif scale_mode == "normalized":
        ax.set_xlabel("Relative position (% of protein length)", labelpad=10)
        ax.set_xticks([0, 25, 50, 75, 100])
    else:
        ax.set_xticks([])
        ax.set_xlabel("")

    ax.set_yticks([])
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    if scale_mode == "equal":
        ax.spines["bottom"].set_visible(False)

    if title:
        ax.set_title(title, fontsize=protein_label_fontsize + 2, fontweight="bold")

    if has_legend_content:
        legend_top_y = bottom - (xlabel_in + position_in + legend_gap_in) / fig_height
        _draw_grouped_legend(fig, ax, proteins, domains_by_protein, color_lookup, alpha,
                              edgecolor, linewidth, label_fontsize, area_left=left,
                              area_right=right, area_top_y=legend_top_y, row_gap_in=0.14)

    saved_paths = []
    if save_prefix:
        png_path, jpg_path = save_prefix + ".png", save_prefix + ".jpg"
        # Note: we deliberately do NOT use bbox_inches="tight" here. Matplotlib's
        # tight-bbox calculation only accounts for the axes' "official" legend slot
        # (set via ax.legend()) and ignores extra Legend artists added via
        # ax.add_artist() -- and forcing their inclusion via bbox_extra_artists
        # triggers a separate matplotlib quirk that mis-renders the axes-fraction
        # protein-name labels. Our own absolute-inch margin budget above (now with
        # a REAL measured legend height) already reserves enough room for
        # everything, so a plain save is more robust.
        fig.savefig(png_path, dpi=dpi, transparent=transparent_png)
        fig.savefig(jpg_path, dpi=dpi, facecolor="white")
        saved_paths = [png_path, jpg_path]
        if save_svg:
            # SVG is vector (paths, not pixels), so there's no dpi/antialiasing
            # concern here -- it's what makes it open cleanly in Illustrator,
            # Inkscape, etc. for further editing.
            svg_path = save_prefix + ".svg"
            fig.savefig(svg_path, transparent=transparent_png)
            saved_paths.append(svg_path)
        if auto_download and IN_COLAB:
            for p in saved_paths:
                colab_files.download(p)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, ax, saved_paths


In [10]:
# 6) Interactive control panel (ipywidgets)
# Run this cell, then use the panel to fetch/add/import proteins, style the figure,
# and click "Plot" to render, then use the Download buttons to save it.
# Re-run this single cell any time you want a fresh panel (e.g. after clearing
# the registry).

def _section(header_text, children, color="#eef3f8"):
    header = widgets.HTML(
        "<div style='background:%s;padding:5px 8px;border-radius:5px;"
        "font-weight:600;margin-bottom:6px;font-size:13px;'>%s</div>" % (color, header_text),
        layout=widgets.Layout(width="100%"))
    return widgets.VBox([header] + list(children),
                         layout=widgets.Layout(border="1px solid #ccc", padding="8px",
                                                margin="5px 0px", width="100%",
                                                max_width="620px", overflow="hidden"))

def _wrap_row(*items):
    # Flex-wrapping row: items flow onto new lines instead of forcing a horizontal
    # scrollbar when the panel is narrower than the sum of the items' widths.
    return widgets.Box(list(items), layout=widgets.Layout(
        display="flex", flex_flow="row wrap", align_items="center",
        width="100%", overflow="hidden"))


feature_checks = {
    ft: widgets.Checkbox(value=True, description=ft, indent=False, style=WSTYLE,
                          layout=widgets.Layout(width="110px"))
    for ft in UNIPROT_FEATURE_TYPES_DEFAULT
}

acc_input = widgets.Text(description="UniProt ID(s):", placeholder="P04637  or  P04637, P00533",
                          style=WSTYLE, layout=widgets.Layout(width="260px"))
acc_name_input = widgets.Text(description="Display name:", placeholder="(only used for a single ID)",
                               style=WSTYLE, layout=widgets.Layout(width="260px"))
fetch_btn = widgets.Button(description="Fetch from UniProt", button_style="primary",
                            layout=widgets.Layout(width="180px"))

manual_name_input = widgets.Text(description="Name:", placeholder="e.g. MyProteinX",
                                  style=WSTYLE, layout=widgets.Layout(width="220px"))
manual_len_input = widgets.IntText(description="Length (aa):", value=300,
                                    style=WSTYLE, layout=widgets.Layout(width="160px"))
manual_add_btn = widgets.Button(description="Add manual protein", button_style="primary",
                                 layout=widgets.Layout(width="180px"))

csv_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Choose CSV",
                                 layout=widgets.Layout(width="180px"))
csv_import_btn = widgets.Button(description="Import CSV", button_style="primary",
                                 layout=widgets.Layout(width="140px"))

domain_protein_dd = widgets.Dropdown(description="Protein:", options=[], style=WSTYLE,
                                      layout=widgets.Layout(width="260px"))
domain_name_input = widgets.Text(description="Domain name:", placeholder="e.g. Kinase domain",
                                  style=WSTYLE, layout=widgets.Layout(width="220px"))
domain_start_input = widgets.IntText(description="Start:", value=1, style=WSTYLE,
                                      layout=widgets.Layout(width="120px"))
domain_end_input = widgets.IntText(description="End:", value=100, style=WSTYLE,
                                    layout=widgets.Layout(width="120px"))
domain_color_input = widgets.ColorPicker(description="Color:", value="#4C72B0", concise=True,
                                          style=WSTYLE)
domain_add_btn = widgets.Button(description="Add domain", button_style="primary",
                                 layout=widgets.Layout(width="140px"))

remove_protein_dd = widgets.Dropdown(description="Remove:", options=[], style=WSTYLE,
                                      layout=widgets.Layout(width="260px"))
remove_btn = widgets.Button(description="Remove protein", button_style="danger",
                             layout=widgets.Layout(width="150px"))
refresh_btn = widgets.Button(description="Refresh list", layout=widgets.Layout(width="130px"))

scale_mode_dd = widgets.Dropdown(description="Scale mode:", options=["true", "equal", "normalized"],
                                  value="true", style=WSTYLE, layout=widgets.Layout(width="200px"))
shape_dd = widgets.Dropdown(description="Domain shape:", options=["rounded", "rect"], value="rounded",
                             style=WSTYLE, layout=widgets.Layout(width="190px"))
palette_dd = widgets.Dropdown(description="Palette:", options=["tab20", "tab20b", "tab20c", "Set2",
                               "Pastel1", "Dark2", "viridis"], value="tab20", style=WSTYLE,
                               layout=widgets.Layout(width="180px"))
corner_slider = widgets.FloatSlider(description="Corner radius:", min=0.0, max=0.5, step=0.05,
                                     value=0.5, style=WSTYLE, layout=widgets.Layout(width="260px"))
domain_height_slider = widgets.FloatSlider(description="Domain height (fraction of the row):", min=0.1,
                                            max=0.95, step=0.05, value=0.6, style=WSTYLE,
                                            layout=widgets.Layout(width="330px"))
backbone_thickness_slider = widgets.FloatSlider(description="Backbone thickness (fraction of domain):",
                                                 min=0.05, max=1.2, step=0.05, value=0.35, style=WSTYLE,
                                                 layout=widgets.Layout(width="380px"))
alpha_slider = widgets.FloatSlider(description="Transparency:", min=0.1, max=1.0, step=0.05,
                                    value=0.9, style=WSTYLE, layout=widgets.Layout(width="260px"))
linewidth_slider = widgets.FloatSlider(description="Edge width:", min=0.0, max=4.0, step=0.25,
                                        value=1.0, style=WSTYLE, layout=widgets.Layout(width="260px"))
edgecolor_picker = widgets.ColorPicker(description="Edge color:", value="#000000", concise=True,
                                        style=WSTYLE)
fig_width_slider = widgets.FloatSlider(description="Fig width (in):", min=6, max=30, step=1,
                                        value=14, style=WSTYLE, layout=widgets.Layout(width="260px"))
row_height_slider = widgets.FloatSlider(description="Row height:", min=0.5, max=3.0, step=0.1,
                                         value=1.0, style=WSTYLE, layout=widgets.Layout(width="260px"))
labels_checkbox = widgets.Checkbox(description="Show domain labels", value=True, style=WSTYLE)
position_labels_checkbox = widgets.Checkbox(description="Show aa position numbers", value=False, style=WSTYLE)
legend_checkbox = widgets.Checkbox(description="Show legend", value=True, style=WSTYLE)
transparent_checkbox = widgets.Checkbox(description="Transparent PNG background", value=True, style=WSTYLE)
title_input = widgets.Text(description="Title:", placeholder="(optional figure title)",
                            style=WSTYLE, layout=widgets.Layout(width="260px"))
filename_input = widgets.Text(description="Save as:", value="protein_domains", style=WSTYLE,
                               layout=widgets.Layout(width="220px"))
dpi_slider = widgets.IntSlider(description="DPI:", min=100, max=1000, step=50, value=300,
                                style=WSTYLE, layout=widgets.Layout(width="260px"))

def _on_shape_change(change):
    # Corner radius has no effect on rectangular domains, so disable it then --
    # a disabled slider makes it obvious at a glance rather than silently ignored.
    corner_slider.disabled = (change["new"] == "rect")

shape_dd.observe(_on_shape_change, names="value")
_on_shape_change({"new": shape_dd.value})



# Feature-type filter checkboxes: rebuilt on the fly (see _refresh_type_filters)
# from whatever dtypes are actually present across the registered proteins, so
# it always matches reality instead of a fixed, possibly-stale list.
TYPE_FILTER_CHECKS = {}
type_filter_row = widgets.Box([widgets.HTML("<i>(fetch or add a protein to see feature types here)</i>")],
                               layout=widgets.Layout(display="flex", flex_flow="row wrap",
                                                      align_items="center", width="100%", overflow="hidden"))

plot_btn = widgets.Button(description="Plot", button_style="success", icon="paint-brush",
                           layout=widgets.Layout(width="120px"))
download_png_btn = widgets.Button(description="Download PNG", icon="download",
                                   layout=widgets.Layout(width="150px"))
download_jpg_btn = widgets.Button(description="Download JPG", icon="download",
                                   layout=widgets.Layout(width="150px"))
download_csv_btn = widgets.Button(description="Download CSV for quick modifications later", icon="download",
                                   layout=widgets.Layout(width="300px"))
download_svg_btn = widgets.Button(description="Download SVG", icon="download",
                                   layout=widgets.Layout(width="150px"))
reset_btn = widgets.Button(description="Reset to default parameters", icon="undo",
                            layout=widgets.Layout(width="220px"))
out = widgets.Output()

LAST_SAVED = {"png": None, "jpg": None, "svg": None, "csv": None}

# Snapshot every style widget's starting value, so "Reset to default parameters"
# can restore it later without hardcoding the values a second time.
_STYLE_WIDGET_DEFAULTS = [
    (scale_mode_dd, scale_mode_dd.value),
    (shape_dd, shape_dd.value),
    (palette_dd, palette_dd.value),
    (corner_slider, corner_slider.value),
    (domain_height_slider, domain_height_slider.value),
    (backbone_thickness_slider, backbone_thickness_slider.value),
    (alpha_slider, alpha_slider.value),
    (linewidth_slider, linewidth_slider.value),
    (edgecolor_picker, edgecolor_picker.value),
    (fig_width_slider, fig_width_slider.value),
    (row_height_slider, row_height_slider.value),
    (labels_checkbox, labels_checkbox.value),
    (position_labels_checkbox, position_labels_checkbox.value),
    (legend_checkbox, legend_checkbox.value),
    (transparent_checkbox, transparent_checkbox.value),
    (title_input, title_input.value),
    (filename_input, filename_input.value),
    (dpi_slider, dpi_slider.value),
]


def _refresh_dropdowns():
    options = list(PROTEINS.keys())
    domain_protein_dd.options = options
    remove_protein_dd.options = options

    # Keep one checkbox per distinct feature dtype actually present right now
    # (default Domain/Region/Motif/Repeat show up as soon as anything of that
    # type exists; a custom dtype typed into section 2 shows up too). Existing
    # checkboxes keep their current on/off state; only genuinely new types get
    # a fresh (checked) checkbox, and types no longer present are dropped.
    present_types = []
    for p in PROTEINS.values():
        for d in p.domains:
            if d.dtype not in present_types:
                present_types.append(d.dtype)

    for dt in list(TYPE_FILTER_CHECKS.keys()):
        if dt not in present_types:
            del TYPE_FILTER_CHECKS[dt]
    for dt in present_types:
        if dt not in TYPE_FILTER_CHECKS:
            TYPE_FILTER_CHECKS[dt] = widgets.Checkbox(value=True, description=dt, indent=False,
                                                       style=WSTYLE, layout=widgets.Layout(width="130px"))

    if present_types:
        type_filter_row.children = [TYPE_FILTER_CHECKS[dt] for dt in present_types]
    else:
        type_filter_row.children = [widgets.HTML("<i>(fetch or add a protein to see feature types here)</i>")]


def _on_fetch_clicked(b):
    with out:
        clear_output()
        chosen_types = [ft for ft, cb in feature_checks.items() if cb.value]
        fetched, errors = fetch_uniprot_multi(acc_input.value, custom_name=acc_name_input.value or None,
                                               feature_types=chosen_types)
        for p in fetched:
            print("Fetched:", p)
        for acc, err in errors:
            print("Error fetching %s: %s" % (acc, err))
        if not fetched and not errors:
            print("Enter at least one UniProt ID.")
        _refresh_dropdowns()


def _on_manual_add_clicked(b):
    with out:
        clear_output()
        if not manual_name_input.value:
            print("Please enter a protein name.")
            return
        p = add_protein_manual(manual_name_input.value, manual_len_input.value)
        print("Added:", p)
        _refresh_dropdowns()


def _on_csv_import_clicked(b):
    with out:
        clear_output()
        val = csv_upload.value
        if not val:
            print("Choose a CSV file first.")
            return
        item = next(iter(val.values())) if isinstance(val, dict) else val[0]
        content = item["content"]
        if isinstance(content, memoryview):
            content = content.tobytes()
        try:
            names = import_domains_csv(content, is_content=True)
            print("Imported/updated:", ", ".join(names) if names else "(no rows found)")
            _refresh_dropdowns()
        except Exception as e:
            print("Error importing CSV:", e)


def _on_domain_add_clicked(b):
    with out:
        clear_output()
        if not domain_protein_dd.value:
            print("Add/fetch a protein first.")
            return
        d = add_domain_to_protein(domain_protein_dd.value, domain_name_input.value or "Domain",
                                   domain_start_input.value, domain_end_input.value,
                                   color=domain_color_input.value)
        print("Added domain to %s: %s" % (domain_protein_dd.value, d))
        _refresh_dropdowns()


def _on_remove_clicked(b):
    with out:
        clear_output()
        if remove_protein_dd.value:
            remove_protein(remove_protein_dd.value)
            print("Removed:", remove_protein_dd.value)
            _refresh_dropdowns()


def _on_refresh_clicked(b):
    with out:
        clear_output()
        list_proteins()
    _refresh_dropdowns()


def _on_plot_clicked(b):
    with out:
        clear_output()
        if not PROTEINS:
            print("No proteins registered yet — fetch, add, or import one first.")
            return
        base = filename_input.value or "protein_domains"
        visible_types = [dt for dt, cb in TYPE_FILTER_CHECKS.items() if cb.value] if TYPE_FILTER_CHECKS else None
        _, _, saved = plot_proteins(
            scale_mode=scale_mode_dd.value,
            domain_shape=shape_dd.value,
            corner_radius=corner_slider.value,
            domain_height=domain_height_slider.value,
            backbone_height_frac=backbone_thickness_slider.value,
            palette=palette_dd.value,
            alpha=alpha_slider.value,
            edgecolor=edgecolor_picker.value,
            linewidth=linewidth_slider.value,
            fig_width=fig_width_slider.value,
            row_height=row_height_slider.value,
            show_domain_labels=labels_checkbox.value,
            show_position_labels=position_labels_checkbox.value,
            visible_types=visible_types,
            show_legend=legend_checkbox.value,
            title=title_input.value or None,
            dpi=dpi_slider.value,
            transparent_png=transparent_checkbox.value,
            save_svg=True,
            save_prefix=base,
            auto_download=False,
        )
        if len(saved) >= 2:
            LAST_SAVED["png"], LAST_SAVED["jpg"] = saved[0], saved[1]
        if len(saved) >= 3:
            LAST_SAVED["svg"] = saved[2]
        csv_path = export_domains_csv(base + ".csv")
        LAST_SAVED["csv"] = csv_path
        print("Saved:", ", ".join(saved + [csv_path]) if saved else csv_path)


def _download(fmt):
    with out:
        path = LAST_SAVED.get(fmt)
        if not path:
            print("Click 'Plot' first.")
            return
        if IN_COLAB:
            colab_files.download(path)
        else:
            print("Saved at:", path, "(auto-download only works inside Google Colab)")


def _on_reset_clicked(b):
    with out:
        clear_output()
        for widget, default in _STYLE_WIDGET_DEFAULTS:
            widget.value = default
        for cb in TYPE_FILTER_CHECKS.values():
            cb.value = True
        print("Style & export settings reset to their defaults.")


download_png_btn.on_click(lambda b: _download("png"))
download_jpg_btn.on_click(lambda b: _download("jpg"))
download_csv_btn.on_click(lambda b: _download("csv"))
download_svg_btn.on_click(lambda b: _download("svg"))
reset_btn.on_click(_on_reset_clicked)
fetch_btn.on_click(_on_fetch_clicked)
manual_add_btn.on_click(_on_manual_add_clicked)
csv_import_btn.on_click(_on_csv_import_clicked)
domain_add_btn.on_click(_on_domain_add_clicked)
remove_btn.on_click(_on_remove_clicked)
refresh_btn.on_click(_on_refresh_clicked)
plot_btn.on_click(_on_plot_clicked)

fetch_box = _section("1a. Fetch protein(s) from UniProt", [
    _wrap_row(acc_input, acc_name_input),
    _wrap_row(*feature_checks.values()),
    fetch_btn,
], color="#e8f0fe")

manual_box = _section("1b. ...or add a protein manually", [
    _wrap_row(manual_name_input, manual_len_input),
    manual_add_btn,
], color="#e6f4ea")

csv_box = _section("1c. ...or import proteins from a CSV", [
    _wrap_row(csv_upload, csv_import_btn),
], color="#e6f4ea")

domain_box = _section("2. Add a custom domain to any registered protein", [
    domain_protein_dd,
    _wrap_row(domain_name_input, domain_start_input, domain_end_input, domain_color_input),
    domain_add_btn,
], color="#fef7e0")

manage_box = _section("Manage registry", [
    _wrap_row(refresh_btn, remove_protein_dd, remove_btn),
], color="#f1f3f4")

style_box = _section("3. Style & export", [
    _wrap_row(scale_mode_dd, shape_dd, palette_dd),
    widgets.HTML("<b style='font-size:12px;'>Show feature types:</b>"),
    type_filter_row,
    _wrap_row(corner_slider),
    _wrap_row(domain_height_slider),
    _wrap_row(backbone_thickness_slider),
    _wrap_row(alpha_slider),
    _wrap_row(linewidth_slider),
    _wrap_row(edgecolor_picker, fig_width_slider),
    _wrap_row(row_height_slider),
    _wrap_row(labels_checkbox, position_labels_checkbox),
    _wrap_row(legend_checkbox, transparent_checkbox),
    _wrap_row(title_input, filename_input, dpi_slider),
    _wrap_row(plot_btn, download_png_btn, download_jpg_btn, download_svg_btn, download_csv_btn),
    _wrap_row(reset_btn),
], color="#fce8e6")

display(widgets.VBox([fetch_box, manual_box, csv_box, domain_box, manage_box, style_box, out],
                      layout=widgets.Layout(width="100%", max_width="640px")))


## Quick start (script it yourself)

You don't need the widget panel — everything above is a plain Python API. Uncomment and run:


In [7]:
# --- Example A: two real proteins fetched from UniProt (comma-separated IDs work too) ---
# fetch_uniprot_multi("P04637, P00533")   # TP53 + EGFR in one call
# plot_proteins(scale_mode="true", domain_shape="rounded", corner_radius=0.5,
#               show_position_labels=True, save_prefix="p53_vs_egfr")

# --- Example B: a manual protein with hand-picked domains, thinner boxes than rows ---
# add_protein_manual("MyProteinX", 450)
# add_domain_to_protein("MyProteinX", "Kinase domain", 20, 280, color="#4C72B0")
# add_domain_to_protein("MyProteinX", "SH2", 300, 390, color="#DD8452", shape="rect")
# plot_proteins(protein_names=["MyProteinX"], scale_mode="equal", domain_height=0.35)

# --- Example C: compare all registered proteins on a normalized (%) scale ---
# plot_proteins(scale_mode="normalized", title="Domain comparison",
#               save_prefix="normalized_comparison")

# --- Example D: export everything to CSV, then re-import it later ---
# export_domains_csv("my_domains.csv")
# clear_registry()
# import_domains_csv("my_domains.csv")

# --- Example E: only show Domain + Repeat features, hiding Region/Motif for this plot ---
# plot_proteins(visible_types=["Domain", "Repeat"])

# --- Example F: the same protein name in two species is auto-disambiguated ---
# fetch_uniprot("P68363")   # e.g. Tubulin alpha chain, human
# fetch_uniprot("P05213")   # e.g. Tubulin alpha chain, a different species
# list_proteins()           # both are kept, each suffixed with its organism

# --- Example G: skip the vector SVG export if you only want PNG/JPG (it's on by default) ---
# plot_proteins(save_prefix="quick_raster_only", save_svg=False)
